# DINOv2/v3, Swin y DETR — Práctica
| IA 5.2 Computer Vision
| FCEIA - UNR

Teoría: [`colab/class_5_Dino_Detr_and_Swin.ipynb`](../colab/class_5_Dino_Detr_and_Swin.ipynb) · [Abrir en Colab](https://colab.research.google.com/github/EzequielMatiasArevalo/tuia-computer-vision/blob/main/colab/class_5_Dino_Detr_and_Swin.ipynb)

En la teoría vimos tres ideas que llevaron los Transformers de visión a la práctica:

| Modelo | Idea central | Qué vamos a hacer con él |
|---|---|---|
| **DINOv2 / DINOv3** | Aprendizaje **auto-supervisado**: features buenas sin etiquetas | Segmentar sin etiquetas (PCA, K-Means), matchear partes entre imágenes, entrenar una cabeza propia |
| **Registers** | Tokens "basura" extra para que la atención no ensucie los patches | Comparar DINOv2 con y sin registers: normas de los tokens y mapas de atención |
| **Swin** | Atención por **ventanas** + ventanas **desplazadas** + jerarquía | Visualizar W-MSA / SW-MSA, la máscara del *cyclic shift*, el costo computacional y hacer fine-tuning |
| **DETR** | Detección como **predicción de conjuntos** | 100 *object queries*, matching Húngaro a mano, especialización de queries y segmentación panóptica |

**Contenido**

1. [DINOv2 como extractor de features](#1)
2. [Fine-tuning de DINOv2 con una cabeza propia](#2)
3. [Registers y DINOv3](#3)
4. [Swin Transformer: ventanas, desplazamiento y fine-tuning](#4)
5. [DETR: object queries, matching Húngaro y segmentación panóptica](#5)
6. [Ejercicios](#6)
7. [Conclusiones](#7)

> **Necesita GPU.** En Colab: *Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU*.
> Todo el cuaderno corre en una T4 en unos ~25 minutos. En CPU las secciones 1, 3 y 5 andan (lentas);
> los entrenamientos de las secciones 2 y 4 se vuelven muy lentos, así que en ese caso bajá `EPOCHS_*`.
>
> **Datos:** no hace falta subir nada. Las imágenes se bajan de COCO val2017 por URL y el dataset de
> entrenamiento (Oxford Flowers-102, splits `train` + `validation`, ~830 MB) se baja de un espejo en
> Hugging Face. Todos los pesos y datos son públicos (no hace falta token de Hugging Face).

---
## Instalación

In [ ]:
# torch, torchvision, matplotlib, scikit-learn y scipy ya vienen en Colab.
# NO reinstalamos torch: un `pip install torch` sin versión puede reemplazar la build con CUDA
# de Colab por una solo-CPU y matar la GPU en silencio.
# timm >= 1.0.20 es el mínimo para los pesos de DINOv3.
!pip install -q "timm>=1.0.20" "transformers>=4.56" datasets

## Librerías

In [ ]:
import io
import math
import time
import random
import urllib.request

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

import torchvision
import torchvision.transforms as T
from torchvision.models import swin_t, Swin_T_Weights
from torchvision.ops import generalized_box_iou, box_convert

import timm
import transformers
from datasets import load_dataset
from transformers import (
    AutoImageProcessor,
    Dinov2Model,
    Dinov2WithRegistersModel,
    DetrForObjectDetection,
    DetrForSegmentation,
)

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.neighbors import KNeighborsClassifier
from scipy.optimize import linear_sum_assignment

import warnings
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch:      {torch.__version__}")
print(f"Torchvision:  {torchvision.__version__}")
print(f"timm:         {timm.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"Dispositivo:  {device}" + ("" if device.type == "cuda" else "  ⚠ sin GPU: bajá EPOCHS_* en la celda de configuración"))

## Configuración

Todos los "perillas" del cuaderno están acá, para no tener que buscarlas celda por celda.

In [ ]:
# ── Resoluciones ──────────────────────────────────────────────────────────────
DINO_SIZE  = 448      # múltiplo de 14 (patch de DINOv2) → grilla de 32×32 patches
CLS_SIZE   = 224      # resolución para clasificación (secciones 2 y 4)

# ── Entrenamiento (secciones 2 y 4) ───────────────────────────────────────────
EPOCHS_HEAD   = 100   # cabeza sobre features congeladas: es muy barato, 1020 vectores
EPOCHS_FT     = 4     # fine-tuning parcial de DINOv2 (últimos bloques)
EPOCHS_SWIN   = 4     # fine-tuning completo de Swin-T
BATCH_SIZE    = 32
TRAIN_MAX_BATCHES = None  # None = todas; un número chico sirve para probar rápido

NUM_WORKERS = 2

# ── Normalización de ImageNet (la usan DINOv2, Swin y DETR) ───────────────────
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

In [ ]:
# ── Helpers de imágenes ───────────────────────────────────────────────────────
COCO_URL = "http://images.cocodataset.org/val2017/{:012d}.jpg"

def cargar_coco(img_id):
    # Baja una imagen de COCO val2017 por su id. Devuelve PIL RGB (o None si falla la red).
    try:
        req = urllib.request.Request(COCO_URL.format(img_id), headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=15) as r:
            return Image.open(io.BytesIO(r.read())).convert("RGB")
    except Exception as e:
        print(f"  no se pudo bajar {img_id}: {e}")
        return None

def desnormalizar(t):
    # Tensor normalizado (3,H,W) → array HxWx3 en [0,1] para mostrar.
    m = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    s = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (t.cpu() * s + m).clamp(0, 1).permute(1, 2, 0).numpy()

def preparar(img, size):
    # PIL → tensor (1,3,size,size) normalizado. Resize cuadrado: simple y la grilla de patches queda exacta.
    tf = T.Compose([
        T.Resize((size, size), interpolation=T.InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    return tf(img).unsqueeze(0)

# Imágenes de COCO val2017 que usamos a lo largo del cuaderno
IDS_SEGMENTACION = {"gatos": 39769, "oso": 285, "pare": 724, "ositos": 776, "esquiadora": 785, "colectivo": 1584}
ID_GATO_2 = 1675

imgs_seg = {k: cargar_coco(v) for k, v in IDS_SEGMENTACION.items()}
imgs_seg = {k: v for k, v in imgs_seg.items() if v is not None}
img_gato_2 = cargar_coco(ID_GATO_2)

fig, axs = plt.subplots(1, len(imgs_seg), figsize=(3 * len(imgs_seg), 3))
for ax, (nombre, img) in zip(axs, imgs_seg.items()):
    ax.imshow(img); ax.set_title(nombre); ax.axis("off")
plt.suptitle("Imágenes de COCO val2017 para las secciones 1 y 3")
plt.tight_layout(); plt.show()

---
<a id="1"></a>
# 1. DINOv2 como extractor de features

DINOv2 se entrenó **sin una sola etiqueta**: un *student* aprende a imitar a un *teacher* (que es un promedio
móvil de sí mismo) sobre distintos recortes de la misma imagen, más una pérdida tipo iBOT sobre patches
enmascarados. Resultado: cada **patch token** termina describiendo "qué parte de qué objeto" hay en ese
pedacito de imagen.

```
Imagen 448×448  →  grilla 32×32 de patches de 14×14  →  ViT  →  [CLS | 1024 patch tokens]  (cada uno de 384-d en ViT-S)
                                                                  ↑         ↑
                                                       imagen entera   una feature por patch → mapas densos
```

En esta sección no entrenamos nada: usamos los patch tokens tal cual salen del modelo para
**separar objeto/fondo**, **colorear partes** y **matchear partes entre imágenes distintas**.

In [ ]:
# DINOv2 ViT-S/14 (21M parámetros) desde Hugging Face
dinov2_s = Dinov2Model.from_pretrained("facebook/dinov2-small").to(device).eval()
PATCH_V2 = dinov2_s.config.patch_size        # 14
print(f"patch size: {PATCH_V2} | dim: {dinov2_s.config.hidden_size} | capas: {dinov2_s.config.num_hidden_layers}")

@torch.no_grad()
def patch_tokens_v2(model, x, n_prefix=1):
    # Devuelve los patch tokens normalizados (B, N, D), sacando CLS (y registers si los hay).
    out = model(pixel_values=x.to(device)).last_hidden_state
    return out[:, n_prefix:, :].float().cpu()

nombres = list(imgs_seg.keys())
batch = torch.cat([preparar(imgs_seg[n], DINO_SIZE) for n in nombres])
feats = patch_tokens_v2(dinov2_s, batch)            # (B, 1024, 384)
B, N, D = feats.shape
G = int(math.isqrt(N))                              # 32
print(f"features: {tuple(feats.shape)} → {B} imágenes, grilla {G}×{G}, {D} dimensiones por patch")

## 1.1 Separar objeto y fondo con la primera componente principal

Para cada imagen tomamos sus 1024 patch tokens y les hacemos PCA. Dentro de una misma foto, la dirección de
mayor varianza suele ser "**objeto vs. fondo**": es lo que más cambia entre patches. El signo de la componente
es arbitrario, así que lo orientamos con una heurística simple: **los bordes de la foto suelen ser fondo**.

> **Ojo:** la PCA va **por imagen**. Si la ajustáramos con los patches de todas las fotos juntas, la dirección
> de mayor varianza pasaría a ser "de qué foto es este patch" (ejercicio 1). Y la heurística de los bordes no
> siempre acierta: fijate qué pasa con el cartel de PARE.

In [ ]:
UMBRAL_FG = 0.0   # sobre la PC1 estandarizada; probá moverlo (ejercicio 1)

def pc1_imagen(f, g):
    # PC1 estandarizada de los patches (N, D) de UNA imagen, orientada para que el borde quede negativo.
    z = PCA(n_components=1, random_state=SEED).fit_transform(f.numpy())[:, 0]
    z = ((z - z.mean()) / z.std()).reshape(g, g)
    borde = np.concatenate([z[0], z[-1], z[:, 0], z[:, -1]])
    return -z if borde.mean() > z.mean() else z

pc1 = np.stack([pc1_imagen(feats[i], G) for i in range(B)])   # (B, G, G)
mascara_fg = pc1 > UMBRAL_FG                                    # True = objeto
X = feats.reshape(-1, D).numpy()                                # todos los patches juntos (lo usa 1.2)

fig, axs = plt.subplots(3, B, figsize=(2.6 * B, 8))
for i, n in enumerate(nombres):
    axs[0, i].imshow(imgs_seg[n].resize((DINO_SIZE, DINO_SIZE))); axs[0, i].set_title(n)
    axs[1, i].imshow(pc1[i], cmap="RdBu_r", vmin=-2.5, vmax=2.5)
    axs[2, i].imshow(mascara_fg[i], cmap="gray")
    for a in axs[:, i]: a.axis("off")
axs[1, 0].text(-8, G / 2, "PC1", rotation=90, va="center"); axs[2, 0].text(-8, G / 2, "PC1 > umbral", rotation=90, va="center")
plt.suptitle("DINOv2: primera componente principal de los patch tokens, por imagen (sin etiquetas)")
plt.tight_layout(); plt.show()

## 1.2 Colorear las partes: PCA de 3 componentes sobre el objeto

Ahora nos quedamos **solo con los patches de objeto** y les hacemos PCA a 3 componentes, que usamos como
R, G y B. Lo importante: el PCA se ajusta con los patches de **todas las imágenes juntas**, entonces el mismo
color significa la misma "dirección semántica" en todas las fotos.

> **Mirá:** ¿las cabezas de los animales tienen colores parecidos entre sí? ¿Y las patas? Nadie le dijo al
> modelo qué es una cabeza.

In [ ]:
X_fg = X[mascara_fg.reshape(-1)]
pca3 = PCA(n_components=3, random_state=SEED).fit(X_fg)
print(f"varianza explicada por las 3 componentes: {pca3.explained_variance_ratio_.round(3)}")

rgb = np.zeros((B * N, 3))
rgb[mascara_fg.reshape(-1)] = pca3.transform(X_fg)
lo, hi = rgb[mascara_fg.reshape(-1)].min(0), rgb[mascara_fg.reshape(-1)].max(0)
rgb = np.where(mascara_fg.reshape(-1, 1), (rgb - lo) / (hi - lo + 1e-8), 0).reshape(B, G, G, 3)

fig, axs = plt.subplots(2, B, figsize=(2.6 * B, 5.5))
for i, n in enumerate(nombres):
    axs[0, i].imshow(imgs_seg[n].resize((DINO_SIZE, DINO_SIZE))); axs[0, i].set_title(n)
    axs[1, i].imshow(rgb[i])
    for a in axs[:, i]: a.axis("off")
plt.suptitle("PCA-3 de los patches de objeto (ajustado sobre todas las imágenes a la vez)")
plt.tight_layout(); plt.show()

## 1.3 Segmentación no supervisada con K-Means

Otra forma de agrupar patches: K-Means con `k` grupos sobre los patch tokens de **cada imagen**. Es una
segmentación sin etiquetas: cada cluster es "un tipo de patch". Usamos distancia coseno normalizando antes
los vectores (K-Means usa euclídea, y sobre vectores unitarios ambas ordenan igual).

In [ ]:
K_CLUSTERS = 4

fig, axs = plt.subplots(2, B, figsize=(2.6 * B, 5.5))
for i, n in enumerate(nombres):
    f = F.normalize(feats[i], dim=-1).numpy()
    etiquetas = KMeans(n_clusters=K_CLUSTERS, n_init=10, random_state=SEED).fit_predict(f).reshape(G, G)
    axs[0, i].imshow(imgs_seg[n].resize((DINO_SIZE, DINO_SIZE))); axs[0, i].set_title(n)
    axs[1, i].imshow(imgs_seg[n].resize((DINO_SIZE, DINO_SIZE)))
    axs[1, i].imshow(etiquetas, cmap="tab10", alpha=0.6, extent=(0, DINO_SIZE, DINO_SIZE, 0),
                     interpolation="nearest", vmin=0, vmax=9)
    for a in axs[:, i]: a.axis("off")
plt.suptitle(f"K-Means (k={K_CLUSTERS}) sobre los patch tokens de cada imagen")
plt.tight_layout(); plt.show()

## 1.4 Correspondencias entre dos imágenes distintas

Si los patch tokens describen "qué parte de qué objeto" es cada patch, deberíamos poder **matchear partes**
entre dos fotos de gatos diferentes: ojo con ojo, oreja con oreja. Para cada patch de objeto de la imagen A
buscamos el más parecido (coseno) en la B, y nos quedamos con los **vecinos mutuos** (A→B y B→A coinciden),
que es el mismo filtro que se usa en matching clásico (clase 1b) para descartar falsos positivos.

In [ ]:
N_LINEAS = 25

img_a, img_b = imgs_seg["gatos"], img_gato_2
fa = F.normalize(patch_tokens_v2(dinov2_s, preparar(img_a, DINO_SIZE))[0], dim=-1)   # (N, D)
fb = F.normalize(patch_tokens_v2(dinov2_s, preparar(img_b, DINO_SIZE))[0], dim=-1)

# Nos quedamos con los patches de objeto de cada imagen (misma receta que en 1.1)
fg_a = pc1_imagen(fa, G).reshape(-1) > UMBRAL_FG
fg_b = pc1_imagen(fb, G).reshape(-1) > UMBRAL_FG

sim = fa @ fb.T                                  # (N, N) similitud coseno
sim[~torch.from_numpy(fg_a)] = -1
sim[:, ~torch.from_numpy(fg_b)] = -1
a2b = sim.argmax(1)
b2a = sim.argmax(0)
mutuos = [(i, a2b[i].item()) for i in range(N) if fg_a[i] and b2a[a2b[i]].item() == i]
print(f"{len(mutuos)} correspondencias mutuas (mostramos hasta {N_LINEAS}, las más fuertes)")
mutuos = sorted(mutuos, key=lambda p: -sim[p[0], p[1]].item())[:N_LINEAS]

def centro(idx):
    r, c = divmod(idx, G)
    return (c + 0.5) * PATCH_V2, (r + 0.5) * PATCH_V2

lienzo = np.concatenate([np.array(img_a.resize((DINO_SIZE, DINO_SIZE))),
                         np.array(img_b.resize((DINO_SIZE, DINO_SIZE)))], axis=1)
plt.figure(figsize=(12, 6)); plt.imshow(lienzo)
colores = plt.cm.hsv(np.linspace(0, 1, len(mutuos), endpoint=False))
for (ia, ib), col in zip(mutuos, colores):
    xa, ya = centro(ia); xb, yb = centro(ib)
    plt.plot([xa, xb + DINO_SIZE], [ya, yb], color=col, lw=1.2)
    plt.scatter([xa, xb + DINO_SIZE], [ya, yb], color=col, s=15)
plt.axis("off"); plt.title("Correspondencias DINOv2 (vecinos mutuos) entre dos gatos distintos")
plt.show()

> **Para pensar:** este matching no usa Harris, ni SIFT, ni ninguna geometría (comparalo con la clase 1b).
> Las correspondencias buenas son **semánticas**: cabeza con cabeza, oreja con oreja, aunque el color, la pose
> y la escala sean distintos. Pero no todas son buenas: buscá las que conectan partes distintas. ¿Por qué
> creés que fallan (pista: la imagen A tiene **dos** gatos y la B casi solo muestra la cabeza)? ¿Qué pasaría
> con SIFT en este par de imágenes?

---
<a id="2"></a>
# 2. Fine-tuning de DINOv2 con una cabeza propia

Vamos a clasificar **Oxford Flowers-102**: 102 especies de flores, con **solo 10 imágenes por clase** en
el split de entrenamiento (1020 en total). Es un escenario *few-shot* donde entrenar una CNN de cero es
imposible, y donde se nota la calidad de las features pre-entrenadas.

Tres escalones, de más barato a más caro:

| Escalón | Backbone | Qué se entrena | Costo |
|---|---|---|---|
| **k-NN** | congelado | nada (vota con los vecinos) | segundos |
| **Linear probe** | congelado | una cabeza propia (`nn.Module`) sobre features cacheadas | segundos |
| **Fine-tuning parcial** | últimos bloques descongelados | cabeza + 2 bloques del ViT | minutos |

In [ ]:
tf_eval = T.Compose([
    T.Resize(256, interpolation=T.InterpolationMode.BICUBIC),
    T.CenterCrop(CLS_SIZE),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
tf_train = T.Compose([
    T.RandomResizedCrop(CLS_SIZE, scale=(0.5, 1.0), interpolation=T.InterpolationMode.BICUBIC),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Flowers-102 desde un espejo en Hugging Face. El servidor original de Oxford (el que usa
# torchvision.datasets.Flowers102) suele estar caído o lento, y no queremos depender de él en clase.
# Usamos los splits oficiales: train (10 por clase) para entrenar y validation (10 por clase) para
# evaluar. El split test (6149 imágenes) pesa 2.5 GB, así que lo dejamos afuera.
hf_flores = load_dataset("dpdl-benchmark/oxford_flowers102", split=["train", "validation"])
NOMBRES_FLORES = hf_flores[0].features["label"].names
N_CLASES = len(NOMBRES_FLORES)

class FloresDataset(torch.utils.data.Dataset):
    # Adapta un split de Hugging Face a (tensor, etiqueta) con el transform que le pasemos.
    def __init__(self, hf_split, transform):
        self.hf, self.transform = hf_split, transform
    def __len__(self):
        return len(self.hf)
    def __getitem__(self, i):
        ej = self.hf[i]
        return self.transform(ej["image"].convert("RGB")), ej["label"]

flores_train     = FloresDataset(hf_flores[0], tf_eval)
flores_train_aug = FloresDataset(hf_flores[0], tf_train)
flores_test      = FloresDataset(hf_flores[1], tf_eval)

print(f"train: {len(flores_train)} imágenes ({len(flores_train) // N_CLASES} por clase)")
print(f"eval:  {len(flores_test)} imágenes (split validation)")

fig, axs = plt.subplots(1, 8, figsize=(16, 2.4))
for ax, i in zip(axs, range(0, 80, 10)):
    x, y = flores_train[i]
    ax.imshow(desnormalizar(x)); ax.set_title(NOMBRES_FLORES[y], fontsize=9); ax.axis("off")
plt.suptitle("Flowers-102 (train): 10 imágenes por clase"); plt.tight_layout(); plt.show()

## 2.1 Features congeladas: CLS + promedio de patches

La receta de evaluación lineal de DINOv2 concatena el **token CLS** (resumen global) con el **promedio de
los patch tokens** (más información local). Con ViT-S: 384 + 384 = 768 dimensiones por imagen.

Como el backbone está congelado, extraemos las features **una sola vez** y las guardamos: entrenar la cabeza
después es instantáneo.

In [ ]:
@torch.no_grad()
def extraer_dino(model, ds, n_prefix=1):
    loader = DataLoader(ds, batch_size=64, shuffle=False, num_workers=NUM_WORKERS)
    feats, labels = [], []
    for x, y in loader:
        h = model(pixel_values=x.to(device)).last_hidden_state
        cls, patches = h[:, 0], h[:, n_prefix:].mean(1)
        feats.append(torch.cat([cls, patches], dim=-1).float().cpu())
        labels.append(y)
    return torch.cat(feats), torch.cat(labels)

t0 = time.time()
Xtr, ytr = extraer_dino(dinov2_s, flores_train)
Xte, yte = extraer_dino(dinov2_s, flores_test)
print(f"features train {tuple(Xtr.shape)} | test {tuple(Xte.shape)} | {time.time() - t0:.0f}s")

# Escalón 0: k-NN con distancia coseno. Cero entrenamiento.
knn = KNeighborsClassifier(n_neighbors=10, metric="cosine").fit(Xtr.numpy(), ytr.numpy())
acc_knn = (knn.predict(Xte.numpy()) == yte.numpy()).mean()
print(f"k-NN (k=10) sobre features DINOv2 congeladas: {acc_knn:.1%}")

## 2.2 La cabeza propia

Una cabeza mínima pero con las piezas que importan:

- `LayerNorm`: CLS y promedio de patches viven en escalas distintas; normalizar estabiliza el entrenamiento.
- `Dropout`: con 10 ejemplos por clase, sobreajustar es lo primero que va a pasar.
- `Linear`: de 768 a 102 clases.

In [ ]:
class CabezaDino(nn.Module):
    # Cabeza de clasificación sobre [CLS ‖ mean(patches)].
    def __init__(self, in_dim, n_clases, p_drop=0.3):
        super().__init__()
        self.norm = nn.LayerNorm(in_dim)
        self.drop = nn.Dropout(p_drop)
        self.fc   = nn.Linear(in_dim, n_clases)

    def forward(self, x):
        return self.fc(self.drop(self.norm(x)))


def entrenar_cabeza(Xtr, ytr, Xte, yte, epochs=EPOCHS_HEAD, lr=1e-3):
    cabeza = CabezaDino(Xtr.shape[1], N_CLASES).to(device)
    opt = torch.optim.AdamW(cabeza.parameters(), lr=lr, weight_decay=1e-2)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    Xtr_d, ytr_d, Xte_d, yte_d = Xtr.to(device), ytr.to(device), Xte.to(device), yte.to(device)
    hist = {"train_loss": [], "test_acc": []}
    for ep in range(epochs):
        cabeza.train()
        perm = torch.randperm(len(Xtr_d), device=device)
        for i in range(0, len(perm), 128):
            b = perm[i:i + 128]
            loss = F.cross_entropy(cabeza(Xtr_d[b]), ytr_d[b], label_smoothing=0.1)
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()
        cabeza.eval()
        with torch.no_grad():
            acc = (cabeza(Xte_d).argmax(1) == yte_d).float().mean().item()
        hist["train_loss"].append(loss.item()); hist["test_acc"].append(acc)
    return cabeza, hist

t0 = time.time()
cabeza_dino, hist_head = entrenar_cabeza(Xtr, ytr, Xte, yte)
acc_probe = hist_head["test_acc"][-1]
print(f"Linear probe DINOv2-S: {acc_probe:.1%} en eval ({time.time() - t0:.1f}s de entrenamiento)")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(hist_head["train_loss"]); ax[0].set_title("loss de train"); ax[0].set_xlabel("época")
ax[1].plot(hist_head["test_acc"]); ax[1].axhline(acc_knn, ls="--", c="gray", label=f"k-NN {acc_knn:.1%}")
ax[1].set_title("accuracy en eval"); ax[1].set_xlabel("época"); ax[1].legend()
plt.tight_layout(); plt.show()

## 2.3 Fine-tuning parcial: descongelar los últimos bloques

Ahora el backbone participa del entrenamiento, pero con cuidado:

- Solo descongelamos los **últimos 2 bloques** (de 12) y la `LayerNorm` final: las primeras capas
  (bordes, texturas) ya son genéricas; las últimas son las más específicas de la tarea original.
- Usamos un **learning rate 10× más chico** para el backbone que para la cabeza: queremos *ajustar*, no
  borrar lo que aprendió en 142M de imágenes.
- Data augmentation (`RandomResizedCrop` + flip), porque ahora la imagen pasa por la red en cada época.

In [ ]:
class DinoClasificador(nn.Module):
    def __init__(self, backbone, cabeza, n_prefix=1):
        super().__init__()
        self.backbone, self.cabeza, self.n_prefix = backbone, cabeza, n_prefix

    def forward(self, x):
        h = self.backbone(pixel_values=x).last_hidden_state
        return self.cabeza(torch.cat([h[:, 0], h[:, self.n_prefix:].mean(1)], dim=-1))


def entrenar(model, train_ds, test_ds, epochs, param_groups, nombre):
    # Loop genérico de entrenamiento con AMP (en GPU). Devuelve el historial.
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, drop_last=True)
    test_loader  = DataLoader(test_ds,  batch_size=64, shuffle=False, num_workers=NUM_WORKERS)
    opt = torch.optim.AdamW(param_groups, weight_decay=0.05)
    total = epochs * (len(train_loader) if TRAIN_MAX_BATCHES is None else min(TRAIN_MAX_BATCHES, len(train_loader)))
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[g["lr"] for g in param_groups], total_steps=total, pct_start=0.15)
    usar_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=usar_amp)
    hist = {"train_loss": [], "test_acc": []}
    for ep in range(epochs):
        model.train(); t0 = time.time(); perdidas = []
        for bi, (x, y) in enumerate(train_loader):
            if TRAIN_MAX_BATCHES is not None and bi >= TRAIN_MAX_BATCHES:
                break
            x, y = x.to(device), y.to(device)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=usar_amp):
                loss = F.cross_entropy(model(x), y, label_smoothing=0.1)
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            perdidas.append(loss.item())
        acc = evaluar(model, test_loader)
        hist["train_loss"].append(float(np.mean(perdidas))); hist["test_acc"].append(acc)
        print(f"[{nombre}] época {ep + 1}/{epochs} | loss {hist['train_loss'][-1]:.3f} | test acc {acc:.1%} | {time.time() - t0:.0f}s")
    return hist


@torch.no_grad()
def evaluar(model, loader):
    model.eval(); ok = n = 0
    for x, y in loader:
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
            pred = model(x.to(device)).argmax(1).cpu()
        ok += (pred == y).sum().item(); n += len(y)
    return ok / n


# Backbone nuevo (no tocamos dinov2_s, que se sigue usando más adelante)
backbone_ft = Dinov2Model.from_pretrained("facebook/dinov2-small")
for p in backbone_ft.parameters():
    p.requires_grad = False
for p in list(backbone_ft.encoder.layer[-2:].parameters()) + list(backbone_ft.layernorm.parameters()):
    p.requires_grad = True

cabeza_ft = CabezaDino(2 * backbone_ft.config.hidden_size, N_CLASES)
cabeza_ft.load_state_dict(cabeza_dino.state_dict())   # arrancamos desde la cabeza del linear probe
dino_ft = DinoClasificador(backbone_ft, cabeza_ft).to(device)

n_entrenables = sum(p.numel() for p in dino_ft.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in dino_ft.parameters())
print(f"parámetros entrenables: {n_entrenables / 1e6:.2f}M de {n_total / 1e6:.2f}M ({n_entrenables / n_total:.0%})")

hist_dino_ft = entrenar(
    dino_ft, flores_train_aug, flores_test, EPOCHS_FT,
    [{"params": [p for p in backbone_ft.parameters() if p.requires_grad], "lr": 2e-5},
     {"params": cabeza_ft.parameters(), "lr": 2e-4}],
    nombre="DINOv2-S FT",
)
acc_dino_ft = hist_dino_ft["test_acc"][-1]

In [ ]:
print(f"{'Método':<35} {'Accuracy eval':>14}")
print("-" * 50)
for nombre, acc in [("k-NN (sin entrenar)", acc_knn),
                    ("Linear probe (cabeza propia)", acc_probe),
                    ("Fine-tuning parcial (2 bloques)", acc_dino_ft)]:
    print(f"{nombre:<35} {acc:>13.1%}")

> **Para pensar:** con features DINOv2, Flowers-102 queda prácticamente **resuelto** ya con k-NN (≈99 %), sin
> entrenar nada. El fine-tuning no tiene margen para mejorar: gastamos minutos de GPU para quedar igual. Esa
> es la promesa de los modelos *foundation*: muchas veces una cabeza lineal (o ni eso) alcanza. Para ver
> dónde sí importan las etiquetas, el ejercicio 2 baja a 1, 2 y 5 imágenes por clase.

---
<a id="3"></a>
# 3. Registers y DINOv3

## 3.1 El problema: tokens con norma gigante

El paper *Vision Transformers Need Registers* (Darcet et al., 2023) encontró algo raro en los ViT grandes:
algunos patch tokens — típicamente en **zonas de fondo poco informativas** — tienen una **norma muchísimo
más grande** que el resto. El modelo los "recicla" como memoria para guardar información global, y eso
ensucia los mapas de atención (aparecen puntos brillantes en el fondo).

La solución: agregar unos pocos tokens extra aprendibles, los **registers**, que no corresponden a ningún
patch. El modelo usa esos tokens como memoria y deja limpios los patches.

```
sin registers:   [CLS | patch_1 ... patch_N]
con registers:   [CLS | reg_1 reg_2 reg_3 reg_4 | patch_1 ... patch_N]   ← los registers se descartan a la salida
```

> **Ojo con la comparación:** DINOv3 **no tiene** versión sin registers (los trae siempre, 4). Para ver el
> efecto aislado comparamos **DINOv2 vs. DINOv2 con registers**, mismo tamaño y mismos datos. Usamos el
> tamaño **Large**: los artefactos aparecen en modelos grandes y casi no se ven en ViT-S.

In [ ]:
# attn_implementation="eager" es necesario para que el modelo devuelva las matrices de atención
dinov2_l     = Dinov2Model.from_pretrained("facebook/dinov2-large", attn_implementation="eager").to(device).eval()
dinov2_l_reg = Dinov2WithRegistersModel.from_pretrained("facebook/dinov2-with-registers-large",
                                                        attn_implementation="eager").to(device).eval()
N_REG = dinov2_l_reg.config.num_register_tokens
print(f"registers en dinov2-with-registers-large: {N_REG}")

@torch.no_grad()
def normas_y_atencion(model, x, n_prefix):
    # Norma de cada patch token en la PENÚLTIMA capa y atención CLS→patches de la última capa.
    # Por qué la penúltima: es la entrada de la última capa, la que "lee" el CLS. Ahí viven los tokens
    # de norma gigante; la salida de la última capa (y la LayerNorm final) ya los aplanan.
    out = model(pixel_values=x.to(device), output_hidden_states=True, output_attentions=True)
    h = out.hidden_states[-2][0]                       # (T, D), salida del bloque 23 de 24
    attn = out.attentions[-1][0].mean(0)               # (T, T), promedio sobre cabezas
    normas = h[n_prefix:].norm(dim=-1).float().cpu()
    cls_attn = attn[0, n_prefix:].float().cpu()
    cls_a_reg = attn[0, 1:n_prefix].sum().item() if n_prefix > 1 else 0.0
    return normas, cls_attn, cls_a_reg

nombres_reg = [n for n in ["gatos", "oso", "esquiadora", "colectivo"] if n in imgs_seg]
resultados = {}
for n in nombres_reg:
    x = preparar(imgs_seg[n], DINO_SIZE)
    resultados[n] = {"sin": normas_y_atencion(dinov2_l, x, 1),
                     "con": normas_y_atencion(dinov2_l_reg, x, 1 + N_REG)}

In [ ]:
# Histograma de normas: ¿aparece una "cola" de tokens con norma enorme?
todas_sin = torch.cat([r["sin"][0] for r in resultados.values()])
todas_con = torch.cat([r["con"][0] for r in resultados.values()])
bins = np.linspace(0, max(todas_sin.max(), todas_con.max()).item(), 80)

plt.figure(figsize=(10, 3.5))
plt.hist(todas_sin.numpy(), bins=bins, alpha=0.6, label="DINOv2-L (sin registers)", log=True)
plt.hist(todas_con.numpy(), bins=bins, alpha=0.6, label="DINOv2-L + 4 registers", log=True)
plt.xlabel("norma del patch token (penúltima capa)"); plt.ylabel("cantidad (escala log)")
plt.legend(); plt.title("Distribución de normas de los patch tokens"); plt.show()

for nombre_m, t in [("sin registers", todas_sin), ("con registers", todas_con)]:
    med = t.median().item()
    print(f"{nombre_m}: mediana {med:.0f} | máximo {t.max():.0f} | tokens con norma > 4× mediana: "
          f"{(t > 4 * med).sum().item()} de {len(t)}")

In [ ]:
fig, axs = plt.subplots(len(nombres_reg), 5, figsize=(16, 3.2 * len(nombres_reg)))
for i, n in enumerate(nombres_reg):
    (nor_s, att_s, _), (nor_c, att_c, reg_c) = resultados[n]["sin"], resultados[n]["con"]
    axs[i, 0].imshow(imgs_seg[n].resize((DINO_SIZE, DINO_SIZE)))
    vmax = max(nor_s.max(), nor_c.max()).item()          # misma escala en ambos mapas
    axs[i, 1].imshow(nor_s.reshape(G, G), cmap="inferno", vmax=vmax); axs[i, 1].set_title("norma — sin reg")
    axs[i, 2].imshow(nor_c.reshape(G, G), cmap="inferno", vmax=vmax); axs[i, 2].set_title("norma — con reg")
    axs[i, 3].imshow(att_s.reshape(G, G), cmap="viridis"); axs[i, 3].set_title("atención CLS — sin reg")
    axs[i, 4].imshow(att_c.reshape(G, G), cmap="viridis"); axs[i, 4].set_title(f"atención CLS — con reg\n({reg_c:.0%} va a los registers)")
    for a in axs[i]: a.axis("off")
plt.suptitle("DINOv2-L: efecto de los registers en las normas y en la atención del CLS", y=1.0)
plt.tight_layout(); plt.show()

> **Mirá:** sin registers, el puntito brillante de la columna "norma" coincide con el punto brillante de la
> atención, y cae **en el fondo** (pasto, nieve, cielo): el CLS le presta atención a esos tokens "memoria" en vez de al
> objeto. Con registers, esa atención se va a los registers (el % del título) y el mapa de los patches queda
> concentrado en los objetos.

## 3.2 DINOv3: misma idea, más escala

DINOv3 (2025) trae de fábrica los 4 registers, **RoPE** como codificación posicional (generaliza mejor a otras
resoluciones), patches de **16×16** y *Gram anchoring* para que las features densas no se degraden en
entrenamientos largos. Lo cargamos con `timm`, que lo expone con la misma API que cualquier otro modelo.

Comparamos la PCA-3 de los patches de DINOv2-S y DINOv3-S sobre las mismas imágenes.

In [ ]:
dinov3_s = timm.create_model("vit_small_patch16_dinov3.lvd1689m", pretrained=True).to(device).eval()
PREFIX_V3 = dinov3_s.num_prefix_tokens          # CLS + registers
PATCH_V3 = 16
cfg_v3 = timm.data.resolve_model_data_config(dinov3_s)
print(f"tokens de prefijo (CLS + registers): {PREFIX_V3} | config de datos: {cfg_v3}")

with torch.no_grad():
    tokens_v3 = dinov3_s.forward_features(preparar(imgs_seg[nombres[0]], DINO_SIZE).to(device))
G3 = DINO_SIZE // PATCH_V3
print(f"forward_features: {tuple(tokens_v3.shape)} = {PREFIX_V3} prefijo + {G3}×{G3} patches")

@torch.no_grad()
def patch_tokens_v3(x):
    return dinov3_s.forward_features(x.to(device))[:, PREFIX_V3:].float().cpu()

def pca_rgb(feats_bnd, g):
    # PCA-3 conjunta de (B, N, D) → (B, g, g, 3) en [0,1]
    b, n, d = feats_bnd.shape
    z = PCA(n_components=3, random_state=SEED).fit_transform(feats_bnd.reshape(-1, d).numpy())
    z = (z - z.min(0)) / (z.max(0) - z.min(0) + 1e-8)
    return z.reshape(b, g, g, 3)

batch = torch.cat([preparar(imgs_seg[n], DINO_SIZE) for n in nombres])
rgb_v2 = pca_rgb(patch_tokens_v2(dinov2_s, batch), G)
rgb_v3 = pca_rgb(patch_tokens_v3(batch), G3)

fig, axs = plt.subplots(3, B, figsize=(2.6 * B, 8))
for i, n in enumerate(nombres):
    axs[0, i].imshow(imgs_seg[n].resize((DINO_SIZE, DINO_SIZE))); axs[0, i].set_title(n)
    axs[1, i].imshow(rgb_v2[i]); axs[2, i].imshow(rgb_v3[i])
    for a in axs[:, i]: a.axis("off")
axs[1, 0].set_title("DINOv2-S/14", loc="left"); axs[2, 0].set_title("DINOv3-S/16", loc="left")
plt.suptitle("PCA-3 de todos los patches (sin máscara de objeto): DINOv2 vs DINOv3")
plt.tight_layout(); plt.show()

## 3.3 DINOv3 con backbone ConvNeXt vs ViT

DINOv3 también se **destiló** a backbones convolucionales (ConvNeXt). Diferencia clave en la forma de las
features:

- **ViT**: una sola grilla de tokens, siempre a la misma resolución (1/16 de la imagen).
- **ConvNeXt**: una **pirámide** de 4 etapas (1/4, 1/8, 1/16, 1/32), como las CNN de la clase 2. Es lo que
  esperan los detectores con FPN.

In [ ]:
convnext_v3 = timm.create_model("convnext_tiny.dinov3_lvd1689m", pretrained=True, features_only=True).to(device).eval()

x = preparar(imgs_seg["gatos"], DINO_SIZE).to(device)
with torch.no_grad():
    etapas = convnext_v3(x)

def contar(m):
    return sum(p.numel() for p in m.parameters()) / 1e6

print(f"ConvNeXt-T DINOv3 ({contar(convnext_v3):.1f}M params) — pirámide de features:")
for i, e in enumerate(etapas):
    print(f"  etapa {i + 1}: {tuple(e.shape)}  → stride {DINO_SIZE // e.shape[-1]}")
print(f"\nViT-S DINOv3 ({contar(dinov3_s):.1f}M params) — una sola grilla: (1, {dinov3_s.embed_dim}, {G3}, {G3}) → stride {PATCH_V3}")

fig, axs = plt.subplots(1, len(etapas) + 2, figsize=(3 * (len(etapas) + 2), 3))
axs[0].imshow(imgs_seg["gatos"].resize((DINO_SIZE, DINO_SIZE))); axs[0].set_title("imagen")
for i, e in enumerate(etapas):
    c, h, w = e.shape[1:]
    axs[i + 1].imshow(pca_rgb(e.permute(0, 2, 3, 1).reshape(1, h * w, c).float().cpu(), h)[0])
    axs[i + 1].set_title(f"ConvNeXt etapa {i + 1}\n{h}×{w}")
axs[-1].imshow(rgb_v3[0]); axs[-1].set_title(f"ViT-S/16\n{G3}×{G3}")
for a in axs: a.axis("off")
plt.suptitle("DINOv3: pirámide ConvNeXt vs grilla única del ViT (PCA-3)"); plt.tight_layout(); plt.show()

In [ ]:
# Liberamos los modelos Large antes de seguir: en una T4 (16 GB) conviene no acumular.
del dinov2_l, dinov2_l_reg
if device.type == "cuda":
    torch.cuda.empty_cache()

---
<a id="4"></a>
# 4. Swin Transformer: ventanas, desplazamiento y fine-tuning

El ViT hace atención **global**: cada patch mira a todos los demás. Con $N = hw$ patches el costo crece como
$N^2$, y eso hace inviable usarlo a alta resolución (detección, segmentación).

Swin propone tres cambios:

1. **W-MSA**: atención solo **dentro de ventanas** de $M \times M$ patches (M = 7). El costo pasa a ser lineal en $N$.
2. **SW-MSA**: en el bloque siguiente, las ventanas se **desplazan** $\lfloor M/2 \rfloor$ patches, para que la
   información cruce de una ventana a otra.
3. **Patch merging**: entre etapas se juntan 2×2 patches → pirámide jerárquica como una CNN.

## 4.1 Ver las ventanas sobre una imagen

Swin-T arranca con patches de **4×4 píxeles**: una imagen de 224×224 → grilla de 56×56 patches → con ventanas
de 7×7 patches quedan **8×8 = 64 ventanas**. En SW-MSA la grilla de ventanas se corre 3 patches (12 píxeles).

In [ ]:
PATCH_SWIN, M = 4, 7
VENTANA_PX = PATCH_SWIN * M            # 28 píxeles
SHIFT_PX = PATCH_SWIN * (M // 2)       # 12 píxeles
img_swin = imgs_seg["oso"].resize((224, 224))

def dibujar_ventanas(ax, offset, titulo):
    ax.imshow(img_swin)
    for k in range(0, 224 + 1, PATCH_SWIN):               # patches (finito)
        ax.axhline(k, color="white", lw=0.2, alpha=0.5); ax.axvline(k, color="white", lw=0.2, alpha=0.5)
    for k in range(-offset, 224 + 1, VENTANA_PX):         # ventanas (grueso)
        if 0 <= k <= 224:
            ax.axhline(k, color="red", lw=2); ax.axvline(k, color="red", lw=2)
    ax.set_xlim(0, 224); ax.set_ylim(224, 0); ax.set_title(titulo); ax.axis("off")

fig, axs = plt.subplots(1, 2, figsize=(11, 5.5))
dibujar_ventanas(axs[0], 0, "W-MSA: ventanas regulares (8×8 ventanas)")
dibujar_ventanas(axs[1], SHIFT_PX, f"SW-MSA: ventanas corridas {M // 2} patches\n(9×9 ventanas, muchas incompletas)")
plt.tight_layout(); plt.show()

## 4.2 El truco del *cyclic shift* y la máscara de atención

Correr las ventanas genera **9×9 = 81 ventanas**, muchas más chicas que 7×7 — calcular eso es ineficiente.
El truco de Swin: **rotar la imagen** (`torch.roll`) para que vuelvan a quedar 8×8 ventanas completas. El
problema: algunas ventanas ahora mezclan pedazos que en la imagen original **no eran vecinos** (el borde
derecho quedó pegado al izquierdo). Para que no se miren entre sí, se usa una **máscara de atención**: −100
(≈ −∞ antes del softmax) entre patches de regiones distintas.

Reproducimos el cálculo de la máscara tal como está en el código de Swin, en una grilla chica para que se vea.

In [ ]:
H_DEMO = W_DEMO = 14          # grilla de 14×14 patches → 2×2 ventanas de 7×7
s = M // 2

# 1) Cada patch recibe el id de la "región" a la que pertenece después del shift (9 regiones)
regiones = torch.zeros(H_DEMO, W_DEMO, dtype=torch.long)
cnt = 0
for hs in (slice(0, -M), slice(-M, -s), slice(-s, None)):
    for ws in (slice(0, -M), slice(-M, -s), slice(-s, None)):
        regiones[hs, ws] = cnt; cnt += 1

# 2) Partimos en ventanas y armamos la máscara: 0 si misma región, -100 si no
def particionar(x, m):
    h, w = x.shape
    return x.view(h // m, m, w // m, m).permute(0, 2, 1, 3).reshape(-1, m * m)

ventanas = particionar(regiones, M)                                # (4 ventanas, 49 patches)
mascara = (ventanas[:, :, None] != ventanas[:, None, :]).float() * -100.0   # (4, 49, 49)

# 3) El cyclic shift sobre una imagen de verdad
img_t = T.ToTensor()(img_swin)
img_roll = torch.roll(img_t, shifts=(-SHIFT_PX, -SHIFT_PX), dims=(1, 2))

fig, axs = plt.subplots(1, 4, figsize=(18, 4.5))
axs[0].imshow(img_roll.permute(1, 2, 0)); axs[0].set_title("imagen tras torch.roll(−12, −12)")
for k in range(0, 225, VENTANA_PX):
    axs[0].axhline(k, color="red", lw=1.5); axs[0].axvline(k, color="red", lw=1.5)
axs[0].axis("off")
axs[1].imshow(regiones, cmap="tab10"); axs[1].set_title("ids de región en una grilla 14×14\n(después del shift)")
for k in (M,):
    axs[1].axhline(k - 0.5, color="k", lw=2); axs[1].axvline(k - 0.5, color="k", lw=2)
for i, w in enumerate([0, 3]):
    axs[2 + i].imshow(mascara[w], cmap="gray", vmin=-100, vmax=0)
    axs[2 + i].set_title(f"máscara 49×49 de la ventana {w}\n(blanco = 0 se miran, negro = −100 no)")
plt.tight_layout(); plt.show()
print("Ventana 0: una sola región → máscara toda en 0 (todos se miran con todos).")
print("Ventana 3 (esquina): mezcla 4 regiones → la máscara bloquea pares de regiones distintas.")
print("El patrón a rayas sale de aplanar la ventana de 7×7 fila por fila: en cada fila hay 4 patches de una región y 3 de otra.")

## 4.3 ¿Cuánto se ahorra? Complejidad de MSA vs W-MSA

De la teoría, para un mapa de $h \times w$ patches con $C$ canales:

$$\Omega(\text{MSA}) = 4hwC^2 + 2(hw)^2C \qquad \Omega(\text{W-MSA}) = 4hwC^2 + 2M^2hwC$$

In [ ]:
C = 96   # canales de la primera etapa de Swin-T
lados = np.array([224, 384, 512, 768, 1024, 1536])
hw = (lados / PATCH_SWIN) ** 2
msa  = 4 * hw * C**2 + 2 * hw**2 * C
wmsa = 4 * hw * C**2 + 2 * M**2 * hw * C

plt.figure(figsize=(8, 4))
plt.plot(lados, msa / 1e9, "o-", label="MSA global (ViT)")
plt.plot(lados, wmsa / 1e9, "o-", label="W-MSA (Swin)")
plt.yscale("log"); plt.xlabel("lado de la imagen (px)"); plt.ylabel("GFLOPs de una capa de atención")
plt.legend(); plt.grid(alpha=0.3); plt.title("Costo de una capa de atención en la primera etapa (C=96, patch 4)")
plt.show()
for l, a, b in zip(lados, msa, wmsa):
    print(f"{l:>5}px: MSA {a / 1e9:9.1f} GFLOPs | W-MSA {b / 1e9:6.1f} GFLOPs | {a / b:6.1f}× más barato")

## 4.4 La jerarquía: Swin se parece a una CNN

Pasamos una imagen por las etapas de Swin-T de `torchvision` y miramos la forma del tensor después de cada
módulo. Ojo: torchvision guarda los mapas como **(B, H, W, C)** (canales al final), no (B, C, H, W).

In [ ]:
swin_pre = swin_t(weights=Swin_T_Weights.IMAGENET1K_V1).to(device).eval()
x = preparar(imgs_seg["oso"], 224).to(device)

print(f"entrada: {tuple(x.shape)}")
with torch.no_grad():
    h = x
    for i, modulo in enumerate(swin_pre.features):
        h = modulo(h)
        tipo = type(modulo).__name__ if not isinstance(modulo, nn.Sequential) else ("PatchEmbed" if i == 0 else "Stage (bloques Swin)")
        print(f"  features[{i}] {tipo:<22} → {tuple(h.shape)}")
print("\nComparar con ViT-B/16: 14×14 tokens de 768-d en TODAS las capas, sin pirámide.")

## 4.5 Fine-tuning de Swin-T en Flowers-102

Mismo dataset y misma evaluación que en la sección 2, así podemos comparar. Acá hacemos fine-tuning
**completo** (todos los parámetros), con el learning rate del backbone más chico que el de la cabeza nueva.

In [ ]:
swin_ft = swin_t(weights=Swin_T_Weights.IMAGENET1K_V1)
swin_ft.head = nn.Linear(swin_ft.head.in_features, N_CLASES)   # 768 → 102
swin_ft = swin_ft.to(device)
print(f"Swin-T: {contar(swin_ft):.1f}M parámetros, todos entrenables")

params_head = list(swin_ft.head.parameters())
ids_head = {id(p) for p in params_head}
params_backbone = [p for p in swin_ft.parameters() if id(p) not in ids_head]

hist_swin = entrenar(
    swin_ft, flores_train_aug, flores_test, EPOCHS_SWIN,
    [{"params": params_backbone, "lr": 1e-4}, {"params": params_head, "lr": 1e-3}],
    nombre="Swin-T FT",
)
acc_swin = hist_swin["test_acc"][-1]

In [ ]:
resumen = [
    ("DINOv2-S k-NN (sin entrenar)",        acc_knn,     "0"),
    ("DINOv2-S linear probe",               acc_probe,   f"{sum(p.numel() for p in cabeza_dino.parameters()) / 1e3:.0f}K"),
    ("DINOv2-S fine-tuning parcial",        acc_dino_ft, f"{n_entrenables / 1e6:.1f}M"),
    ("Swin-T fine-tuning completo",         acc_swin,    f"{contar(swin_ft):.1f}M"),
]
print(f"{'Método':<36} {'Acc eval':>9} {'Params entrenados':>18}")
print("-" * 66)
for n, a, p in resumen:
    print(f"{n:<36} {a:>9.1%} {p:>18}")

plt.figure(figsize=(8, 3.5))
plt.plot(range(1, EPOCHS_FT + 1), hist_dino_ft["test_acc"], "o-", label="DINOv2-S FT parcial")
plt.plot(range(1, EPOCHS_SWIN + 1), hist_swin["test_acc"], "o-", label="Swin-T FT completo")
plt.axhline(acc_probe, ls="--", c="gray", label="DINOv2-S linear probe")
plt.xlabel("época"); plt.ylabel("accuracy eval"); plt.legend(); plt.grid(alpha=0.3); plt.show()

> **Para pensar:** Swin-T se pre-entrenó **con etiquetas** en ImageNet-1k (1.3M imágenes) y DINOv2
> **sin etiquetas** en 142M. ¿Cuál anda mejor con 10 imágenes por clase? ¿Qué te dice eso sobre la
> diferencia entre "supervisado en poco" y "auto-supervisado en mucho"?

---
<a id="5"></a>
# 5. DETR: object queries, matching Húngaro y segmentación panóptica

Faster R-CNN y YOLO producen miles de cajas candidatas y después necesitan **NMS** para limpiar duplicados.
DETR cambia el planteo: el decoder recibe **100 *object queries*** aprendidas, y cada una devuelve
**exactamente una** predicción (clase + caja), donde una de las clases es **∅ "no hay objeto"**. Durante el
entrenamiento, cada objeto real se asigna a **una sola** query con el **algoritmo Húngaro** → no hace falta NMS.

```
imagen → CNN (ResNet-50) → encoder Transformer → decoder Transformer ← 100 object queries
                                                        ↓
                                    100 × (clase ∈ 91 + ∅, caja cx,cy,w,h)
```

In [ ]:
detr_proc  = AutoImageProcessor.from_pretrained("facebook/detr-resnet-50")
detr       = DetrForObjectDetection.from_pretrained("facebook/detr-resnet-50").to(device).eval()
ID2LABEL   = detr.config.id2label
N_QUERIES  = detr.config.num_queries
print(f"object queries: {N_QUERIES} | clases: {len(ID2LABEL)} + ∅")

img_det = imgs_seg["gatos"]
inputs = detr_proc(images=img_det, return_tensors="pt").to(device)
with torch.no_grad():
    out_det = detr(**inputs)

print(f"logits:     {tuple(out_det.logits.shape)}  → (batch, queries, clases + ∅)")
print(f"pred_boxes: {tuple(out_det.pred_boxes.shape)}  → (batch, queries, cx cy w h normalizados)")

probas = out_det.logits.softmax(-1)[0, :, :-1]          # sacamos la columna ∅
score, clase = probas.max(-1)
print(f"\nqueries con score > 0.9: {(score > 0.9).sum().item()} de {N_QUERIES}")

## 5.1 Las 100 predicciones y el umbral de confianza

DETR **siempre** devuelve 100 predicciones. Casi todas deberían decir "∅" (o tener score bajo para todas las
clases reales). El umbral de confianza es lo único que separa esas 100 de las que mostramos.

In [ ]:
UMBRALES = [0.3, 0.5, 0.7, 0.9]

def dibujar_detecciones(ax, img, boxes_xyxy, etiquetas, scores, titulo):
    ax.imshow(img); ax.set_title(titulo); ax.axis("off")
    for (x0, y0, x1, y1), lab, sc in zip(boxes_xyxy, etiquetas, scores):
        ax.add_patch(mpatches.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, color="lime", lw=2))
        ax.text(x0, y0, f"{lab} {sc:.2f}", color="black", fontsize=7, bbox=dict(facecolor="lime", alpha=0.8, pad=1))

fig, axs = plt.subplots(1, len(UMBRALES) + 1, figsize=(4.2 * (len(UMBRALES) + 1), 3.8))
axs[0].hist(score.cpu().numpy(), bins=40, color="steelblue"); axs[0].set_yscale("log")
axs[0].set_title("score máximo de las 100 queries"); axs[0].set_xlabel("score")
for ax, u in zip(axs[1:], UMBRALES):
    res = detr_proc.post_process_object_detection(out_det, threshold=u, target_sizes=[img_det.size[::-1]])[0]
    dibujar_detecciones(ax, img_det, res["boxes"].cpu().numpy(), [ID2LABEL[l.item()] for l in res["labels"]],
                        res["scores"].cpu().numpy(), f"umbral {u}: {len(res['scores'])} cajas")
plt.tight_layout(); plt.show()

## 5.2 El matching Húngaro, a mano

En el entrenamiento hay que decidir **qué query se compara con qué objeto real**. DETR arma una matriz de
costo de 100 queries × M objetos reales y busca la asignación 1 a 1 de **costo total mínimo** (algoritmo
Húngaro, `scipy.optimize.linear_sum_assignment`). El costo combina tres términos:

$$\mathcal{C}(i, j) = -\lambda_{cls}\, \hat p_i(c_j) \;+\; \lambda_{L1} \lVert b_j - \hat b_i \rVert_1 \;-\; \lambda_{giou}\, \text{GIoU}(b_j, \hat b_i)$$

con $\lambda_{cls}=1,\ \lambda_{L1}=5,\ \lambda_{giou}=2$ (los del paper).

> **Sobre el *ground truth*:** para no bajar las anotaciones de COCO (~250 MB) usamos cajas **anotadas a
> mano de forma aproximada** para esta imagen. No son las oficiales de COCO; alcanzan para ver el mecanismo.

In [ ]:
# Ground truth APROXIMADO (anotado a mano) para la imagen 39769: x0, y0, x1, y1 en píxeles (640×480)
GT = [
    ("cat",    (12, 52, 316, 472)),
    ("cat",    (344, 22, 640, 372)),
    ("remote", (38, 70, 176, 118)),
    ("remote", (332, 76, 370, 188)),
    ("couch",  (0, 0, 640, 476)),
]
LABEL2ID = {v: k for k, v in ID2LABEL.items()}
W_img, H_img = img_det.size

gt_labels = torch.tensor([LABEL2ID[c] for c, _ in GT])
gt_xyxy = torch.tensor([b for _, b in GT], dtype=torch.float) / torch.tensor([W_img, H_img, W_img, H_img])
gt_cxcywh = box_convert(gt_xyxy, "xyxy", "cxcywh")

pred_prob = out_det.logits.softmax(-1)[0].cpu()            # (100, 92)
pred_cxcywh = out_det.pred_boxes[0].cpu()                   # (100, 4)
pred_xyxy = box_convert(pred_cxcywh, "cxcywh", "xyxy")

costo_cls  = -pred_prob[:, gt_labels]                                   # (100, M)
costo_l1   = torch.cdist(pred_cxcywh, gt_cxcywh, p=1)                   # (100, M)
costo_giou = -generalized_box_iou(pred_xyxy, gt_xyxy)                   # (100, M)
COSTO = 1 * costo_cls + 5 * costo_l1 + 2 * costo_giou

filas, cols = linear_sum_assignment(COSTO.numpy())
print(f"{'objeto real':<14} {'query':>6} {'costo':>7} {'p(clase)':>9} {'GIoU':>6}")
for q, j in sorted(zip(filas, cols), key=lambda t: t[1]):
    print(f"{GT[j][0]:<14} {q:>6} {COSTO[q, j]:>7.2f} {-costo_cls[q, j]:>9.2f} {-costo_giou[q, j]:>6.2f}")
print(f"\nLas otras {N_QUERIES - len(GT)} queries se asignan a ∅ (\"no hay objeto\").")

In [ ]:
# Visualización: matriz de costo de las queries "interesantes" + asignación sobre la imagen
interesantes = sorted(set(filas.tolist()) | set(torch.topk(score.cpu(), 10).indices.tolist()))
fig, axs = plt.subplots(1, 2, figsize=(16, 5.5), gridspec_kw={"width_ratios": [1, 1.3]})

im = axs[0].imshow(COSTO[interesantes].numpy(), cmap="viridis_r", aspect="auto")
axs[0].set_yticks(range(len(interesantes)), [f"q{q}" for q in interesantes])
axs[0].set_xticks(range(len(GT)), [f"{c}#{j}" for j, (c, _) in enumerate(GT)], rotation=30)
for q, j in zip(filas, cols):
    axs[0].add_patch(mpatches.Rectangle((j - 0.5, interesantes.index(q) - 0.5), 1, 1, fill=False, color="red", lw=2.5))
plt.colorbar(im, ax=axs[0]); axs[0].set_title("Costo (query × objeto). Rojo = asignación Húngara")

axs[1].imshow(img_det); axs[1].axis("off"); axs[1].set_title("GT aproximado (punteado) y query asignada (lleno)")
colores = plt.cm.tab10(np.arange(len(GT)))
for q, j in zip(filas, cols):
    gx0, gy0, gx1, gy1 = GT[j][1]
    axs[1].add_patch(mpatches.Rectangle((gx0, gy0), gx1 - gx0, gy1 - gy0, fill=False, ls="--", lw=2, color=colores[j]))
    px0, py0, px1, py1 = (pred_xyxy[q] * torch.tensor([W_img, H_img, W_img, H_img])).tolist()
    axs[1].add_patch(mpatches.Rectangle((px0, py0), px1 - px0, py1 - py0, fill=False, lw=2, color=colores[j]))
    axs[1].text(px0 + 3, py1 - 6, f"q{q}→{GT[j][0]}", color="white", fontsize=8, bbox=dict(facecolor=colores[j], pad=1))
plt.tight_layout(); plt.show()

## 5.3 ¿Qué aprende cada object query?

Las 100 queries son vectores aprendidos, iguales para todas las imágenes. El paper de DETR mostró que cada
una se **especializa** en ciertas zonas y tamaños de caja. Lo verificamos: pasamos ~24 imágenes de COCO y,
para algunas queries, graficamos el **centro** de la caja que predicen en cada imagen (color = tamaño).

In [ ]:
IDS_QUERIES = [39769, 139, 285, 632, 724, 776, 785, 802, 872, 885, 1000, 1268,
               1296, 1353, 1425, 1490, 1503, 1532, 1584, 1675, 2153, 2261, 2299, 2431]
imgs_q = [im for im in (cargar_coco(i) for i in IDS_QUERIES) if im is not None]

cajas = []        # (n_imgs, 100, 4) en cx cy w h normalizados
scores_q = []
with torch.no_grad():
    for im in imgs_q:
        o = detr(**detr_proc(images=im, return_tensors="pt").to(device))
        cajas.append(o.pred_boxes[0].cpu()); scores_q.append(o.logits.softmax(-1)[0, :, :-1].max(-1).values.cpu())
cajas = torch.stack(cajas); scores_q = torch.stack(scores_q)
print(f"{len(imgs_q)} imágenes procesadas → cajas {tuple(cajas.shape)}")

QUERIES_A_VER = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90]
fig, axs = plt.subplots(2, 5, figsize=(16, 6.5))
for ax, q in zip(axs.flat, QUERIES_A_VER):
    cx, cy, w, h = cajas[:, q].T
    sc = ax.scatter(cx, cy, c=(w * h).sqrt(), cmap="plasma", vmin=0, vmax=1, s=20 + 80 * scores_q[:, q])
    ax.set_xlim(0, 1); ax.set_ylim(1, 0); ax.set_aspect("equal"); ax.set_title(f"query {q}")
    ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(sc, ax=axs, label="tamaño de la caja (√área relativa)", shrink=0.8)
plt.suptitle("Centro de la caja predicha por cada query en todas las imágenes (punto grande = score alto)")
plt.show()

> **Mirá:** cada query tiende a predecir cajas en una **zona** de la imagen y con un **tamaño** típico. No hay
> anchors definidos a mano (clase 3): las queries cumplen ese rol, pero **aprendidas**.

## 5.4 Segmentación panóptica

DETR se extiende a segmentación agregando una cabeza de máscaras sobre la salida del decoder: cada query
predice además una máscara. **Panóptica** = cada píxel recibe una clase, y si es una *cosa* contable
(*thing*: persona, auto) también un id de instancia; si es *materia* (*stuff*: cielo, pasto, pared) solo la clase.

In [ ]:
pan_proc = AutoImageProcessor.from_pretrained("facebook/detr-resnet-50-panoptic")
detr_pan = DetrForSegmentation.from_pretrained("facebook/detr-resnet-50-panoptic").to(device).eval()
ID2LABEL_PAN = dict(detr_pan.config.id2label)

# La config del modelo no trae nombre para las categorías "stuff" fusionadas de COCO panoptic (184-200):
# aparecen como LABEL_184, etc. Completamos los nombres a mano.
STUFF_FUSIONADAS = {
    184: "tree", 185: "fence", 186: "ceiling", 187: "sky", 188: "cabinet", 189: "table",
    190: "floor", 191: "pavement", 192: "mountain", 193: "grass", 194: "dirt", 195: "paper",
    196: "food-other", 197: "building", 198: "rock", 199: "wall", 200: "rug",
}
for k, v in STUFF_FUSIONADAS.items():
    if str(ID2LABEL_PAN.get(k, "LABEL_")).startswith("LABEL_"):
        ID2LABEL_PAN[k] = v

def es_thing(label_id):
    # En COCO panoptic las categorías "thing" tienen id ≤ 90 y las "stuff" id > 90
    return label_id <= 90

IDS_PANOPTICA = [139, 632, 1532, 872]
imgs_pan = [im for im in (cargar_coco(i) for i in IDS_PANOPTICA) if im is not None]

rng = np.random.default_rng(SEED)
fig, axs = plt.subplots(2, len(imgs_pan), figsize=(5 * len(imgs_pan), 8))
for i, im in enumerate(imgs_pan):
    with torch.no_grad():
        o = detr_pan(**pan_proc(images=im, return_tensors="pt").to(device))
    res = pan_proc.post_process_panoptic_segmentation(o, threshold=0.85, target_sizes=[im.size[::-1]])[0]
    seg, info = res["segmentation"].cpu().numpy(), res["segments_info"]

    color = np.zeros((*seg.shape, 3))
    leyenda = []
    for s in info:
        c = rng.random(3)
        color[seg == s["id"]] = c
        nombre = ID2LABEL_PAN[s["label_id"]]
        leyenda.append(mpatches.Patch(color=c, label=f"{nombre} ({'thing' if es_thing(s['label_id']) else 'stuff'})"))
    n_things = sum(es_thing(s["label_id"]) for s in info)

    axs[0, i].imshow(im); axs[0, i].axis("off")
    axs[1, i].imshow(im); axs[1, i].imshow(color, alpha=0.6); axs[1, i].axis("off")
    axs[1, i].set_title(f"{len(info)} segmentos: {n_things} things, {len(info) - n_things} stuff")
    axs[1, i].legend(handles=leyenda[:10], fontsize=7, loc="lower left", framealpha=0.8)
plt.suptitle("DETR panóptico (ResNet-50)"); plt.tight_layout(); plt.show()

---
<a id="6"></a>
# 6. Ejercicios

## Ejercicio 1 — El umbral de la máscara de objeto

En la sección 1.1 separamos objeto y fondo con `PC1 > UMBRAL_FG`.

1. Probá con `UMBRAL_FG` en `[-0.5, 0.0, 0.5, 1.0]` y mostrá las máscaras resultantes para todas las imágenes.
2. ¿Qué imagen es la más sensible al umbral? ¿Por qué creés que pasa (pensá en qué es "fondo" en esa foto)?
3. Repetí la PC1 ajustando la PCA con los patches de **todas las imágenes juntas** (`X`). ¿Por qué falla?
   ¿Qué está capturando ahora la dirección de mayor varianza?

In [ ]:
# Ejercicio 1 — resuelto (puntos 1 y 3)
umbrales_ej1 = [-0.5, 0.0, 0.5, 1.0]

fig, axs = plt.subplots(len(umbrales_ej1), B, figsize=(2.3 * B, 2.3 * len(umbrales_ej1)))
for r, u in enumerate(umbrales_ej1):
    for i in range(B):
        axs[r, i].imshow(pc1[i] > u, cmap="gray"); axs[r, i].axis("off")
        if r == 0: axs[r, i].set_title(nombres[i])
    axs[r, 0].text(-4, G / 2, f"u={u}", rotation=90, va="center", ha="right")
plt.suptitle("Máscara de objeto según el umbral sobre la PC1"); plt.tight_layout(); plt.show()

# Punto 3: PCA ajustada con TODAS las imágenes juntas
pc1_conj = PCA(n_components=1, random_state=SEED).fit_transform(X)[:, 0]
pc1_conj = ((pc1_conj - pc1_conj.mean()) / pc1_conj.std()).reshape(B, G, G)
fig, axs = plt.subplots(2, B, figsize=(2.3 * B, 4.8))
for i in range(B):
    axs[0, i].imshow(pc1[i], cmap="RdBu_r", vmin=-2.5, vmax=2.5); axs[0, i].set_title(nombres[i])
    axs[1, i].imshow(pc1_conj[i], cmap="RdBu_r", vmin=-2.5, vmax=2.5)
    for a in axs[:, i]: a.axis("off")
axs[0, 0].text(-4, G / 2, "por imagen", rotation=90, va="center", ha="right")
axs[1, 0].text(-4, G / 2, "conjunta", rotation=90, va="center", ha="right")
plt.suptitle("PC1 por imagen vs PC1 conjunta"); plt.tight_layout(); plt.show()
print("Media de la PC1 conjunta por imagen:", {n: round(float(pc1_conj[i].mean()), 2) for i, n in enumerate(nombres)})
# COMPLETAR (punto 2): respondé en una celda de texto.

## Ejercicio 2 — ¿Cuántas etiquetas hacen falta?

La pregunta clave del aprendizaje auto-supervisado es **cuántos datos etiquetados necesitás después**.

1. Usando las features ya extraídas (`Xtr`, `ytr`, `Xte`, `yte`), entrená la cabeza con **1, 2, 5 y 10**
   imágenes por clase.
2. Graficá accuracy vs. imágenes por clase, para el linear probe y para k-NN.
3. ¿Con cuántas imágenes por clase llegás al 90 % del resultado con 10?

In [ ]:
# Ejercicio 2 — resuelto
SHOTS = [1, 2, 5, 10]
acc_probe_shots, acc_knn_shots = [], []

rng_ej2 = np.random.default_rng(SEED)
for k in SHOTS:
    idx = np.concatenate([rng_ej2.choice(np.where(ytr.numpy() == c)[0], k, replace=False) for c in range(N_CLASES)])
    # Con menos datos hay menos pasos por época: escalamos las épocas para dar ~el mismo número de pasos
    _, h = entrenar_cabeza(Xtr[idx], ytr[idx], Xte, yte, epochs=EPOCHS_HEAD * 10 // k)
    acc_probe_shots.append(h["test_acc"][-1])
    knn_k = KNeighborsClassifier(n_neighbors=min(k, 10), metric="cosine").fit(Xtr[idx].numpy(), ytr[idx].numpy())
    acc_knn_shots.append((knn_k.predict(Xte.numpy()) == yte.numpy()).mean())
    print(f"{k:>2} img/clase → linear probe {acc_probe_shots[-1]:.1%} | k-NN {acc_knn_shots[-1]:.1%}")

plt.figure(figsize=(7, 4))
plt.plot(SHOTS, acc_probe_shots, "o-", label="linear probe")
plt.plot(SHOTS, acc_knn_shots, "s-", label="k-NN")
plt.axhline(0.9 * acc_probe_shots[-1], ls=":", c="gray", label="90 % del resultado con 10")
plt.xscale("log"); plt.xticks(SHOTS, SHOTS); plt.xlabel("imágenes por clase"); plt.ylabel("accuracy eval")
plt.grid(alpha=0.3); plt.legend(); plt.title("Flowers-102 con features DINOv2-S congeladas"); plt.show()

## Ejercicio 3 — Matching Húngaro vs. asignación golosa

En la sección 5.2 usamos el algoritmo Húngaro. Una alternativa "ingenua" es la **golosa**: para cada objeto
real, en orden, elegir la query de menor costo que todavía esté libre.

1. Implementá `asignacion_golosa(COSTO)` que devuelva `(filas, cols)` como `linear_sum_assignment`.
2. Compará el **costo total** de ambas asignaciones. ¿Puede la golosa ser mejor? ¿Puede ser igual?
3. Probá cambiar el orden en que recorrés los objetos (por ejemplo, empezando por el `couch`). ¿Cambia el resultado?

In [ ]:
# Ejercicio 3 — completar
def asignacion_golosa(costo, orden=None):
    costo = costo.clone()
    orden = list(range(costo.shape[1])) if orden is None else orden
    filas, cols = [], []
    for j in orden:
        # COMPLETAR: elegir la query (fila) libre de menor costo para el objeto j
        q = costo[:, j].argmin().item()
        filas.append(q); cols.append(j)
        costo[q, :] = float("inf")      # la query q ya no está disponible
    return np.array(filas), np.array(cols)

f_g, c_g = asignacion_golosa(COSTO)
print(f"Costo total Húngaro: {COSTO[filas, cols].sum():.3f}")
print(f"Costo total goloso:  {COSTO[f_g, c_g].sum():.3f}")

# COMPLETAR (punto 3): probá asignacion_golosa(COSTO, orden=[4, 0, 1, 2, 3]) y compará

## Ejercicio 4 — Swin: ¿qué cambia con otro tamaño de ventana?

1. Modificá `M` (tamaño de ventana) en la sección 4.3 y graficá el costo de W-MSA para `M ∈ {4, 7, 12, 16}`.
2. Volvé a correr la celda de la máscara (4.2) con `M = 4` y `H_DEMO = W_DEMO = 8`. ¿Cuántas regiones distintas
   hay en la ventana de la esquina?
3. **Pregunta:** si la ventana es más grande, el costo sube pero ¿qué se gana? ¿Qué pasa en el límite en que la
   ventana cubre toda la imagen?

In [ ]:
# Ejercicio 4 — resuelto (punto 1)
plt.figure(figsize=(8, 4))
plt.plot(lados, msa / 1e9, "k--", label="MSA global")
for m in [4, 7, 12, 16]:
    plt.plot(lados, (4 * hw * C**2 + 2 * m**2 * hw * C) / 1e9, "o-", label=f"W-MSA M={m}")
plt.yscale("log"); plt.xlabel("lado de la imagen (px)"); plt.ylabel("GFLOPs"); plt.legend(); plt.grid(alpha=0.3)
plt.title("Costo de W-MSA según el tamaño de ventana"); plt.show()
# COMPLETAR (puntos 2 y 3)

## Ejercicio 5 — DINOv3 con registers: ¿se mantienen limpios los mapas?

En la sección 3.1 vimos los registers en DINOv2. DINOv3 (`dinov3_s`) también trae 4 registers.

1. Usando `dinov3_s.forward_features`, calculá la **norma** de los patch tokens (descartando los primeros
   `PREFIX_V3`) para las imágenes de `nombres_reg`, y mostrala como mapa de `G3 × G3`.
2. ¿Aparecen puntos de norma anómala en el fondo como en DINOv2-L sin registers?
3. **Bonus:** calculá también la norma de los 4 registers. ¿Son más grandes o más chicas que la de los patches?

> Nota: `forward_features` de timm devuelve los tokens **después** de la `LayerNorm` final, lo que achica
> las diferencias de norma. Para un análisis fino se puede usar `dinov3_s.forward_intermediates(...)`.

In [ ]:
# Ejercicio 5 — completar
fig, axs = plt.subplots(2, len(nombres_reg), figsize=(3.2 * len(nombres_reg), 6.5))
for i, n in enumerate(nombres_reg):
    with torch.no_grad():
        tok = dinov3_s.forward_features(preparar(imgs_seg[n], DINO_SIZE).to(device))[0].float().cpu()
    # COMPLETAR: normas de los patches (descartar los PREFIX_V3 primeros) y de los registers (tokens 1..PREFIX_V3-1)
    normas_patches = tok[PREFIX_V3:].norm(dim=-1)
    normas_regs = tok[1:PREFIX_V3].norm(dim=-1)
    axs[0, i].imshow(imgs_seg[n].resize((DINO_SIZE, DINO_SIZE))); axs[0, i].axis("off"); axs[0, i].set_title(n)
    axs[1, i].imshow(normas_patches.reshape(G3, G3), cmap="inferno"); axs[1, i].axis("off")
    axs[1, i].set_title(f"patches μ={normas_patches.mean():.1f}\nregisters μ={normas_regs.mean():.1f}")
plt.suptitle("DINOv3-S: norma de los patch tokens"); plt.tight_layout(); plt.show()

---
<a id="7"></a>
# 7. Conclusiones

| Modelo | Qué vimos | Idea para llevarse |
|---|---|---|
| **DINOv2** | PCA, K-Means y matching sobre patch tokens | Sin una sola etiqueta, las features ya separan objetos y partes |
| **DINOv2 + cabeza** | k-NN, linear probe y fine-tuning parcial en Flowers-102 | Con features buenas, una cabeza lineal y 10 imágenes por clase ya rinden |
| **Registers** | Normas y atención de DINOv2-L con y sin registers | Los ViT grandes "reciclan" patches de fondo como memoria; los registers les dan un lugar propio |
| **DINOv3** | Tokens `[CLS | 4 reg | patches]`, ViT vs ConvNeXt | Misma familia, más escala; la versión ConvNeXt da una pirámide lista para detectores |
| **Swin** | Ventanas, *cyclic shift* + máscara, complejidad, fine-tuning | Atención local + desplazamiento = costo lineal y jerarquía tipo CNN |
| **DETR** | 100 queries, umbral, Húngaro a mano, queries especializadas, panóptica | Detección como conjunto: asignación 1 a 1, sin anchors ni NMS |

**Conceptos clave de esta práctica:**

- Los **patch tokens** de un ViT auto-supervisado son features densas: sirven para segmentar y matchear sin entrenar.
- **Linear probe** (backbone congelado) es la forma estándar de medir qué tan buenas son unas features.
- Los **registers** existen por un problema concreto y medible: tokens con norma anómala que ensucian la atención.
- **W-MSA** baja el costo de cuadrático a lineal; **SW-MSA** recupera la comunicación entre ventanas; la máscara
  hace que el *cyclic shift* sea correcto.
- El **matching Húngaro** es lo que permite a DETR entrenar sin NMS: cada objeto real "le pertenece" a una sola query.

**En la próxima clase** vemos CLIP, SAM y Grounding DINO: modelos que conectan imagen y texto, y que llevan la
idea de "features genéricas" a detección y segmentación por *prompt*.